# Runde 1b · H_tq vorbereiten und K_S1 neu rechnen · H_tq-Variante

**In Konto A öffnen**, in einer frischen Laufzeit. GPU-Notebooks benötigen eine A100; ANA verwendet CPU.

**Post hoc, Änderung 2:** kleinere operative Neuronenmenge H_tq; Fragen, Stichproben, Schlüssel und Schwellen bleiben unverändert. Die bisherige Restschätzung von 15–18 A100-Stunden ist eine alte Planbasis, keine Laufzeitzusage; Umstellung und nötige Neuberechnungen kommen hinzu.

**Zuerst alle alten Runde-1b-Läufe in beiden Konten stoppen.** Dieses Notebook legt die getrennten Variantenordner in A und B an und übernimmt geprüfte, H-unabhängige Ergebnisse. Originale unter `dsse_runs` bleiben erhalten. Für Konto B ist die Verknüpfung `MyDrive/dsse_runs_B` mit Bearbeiterrecht nötig.

Danach rechnet es **nur K_S1 ab den von H abhängigen Schritten neu** und stoppt. Das bereits gerechnete Screening wird übernommen. Erst nach erfolgreichem Abschluss mit den weiteren GPU-Notebooks fortfahren.

**Hidden States mitschneiden:** standardmäßig eingeschaltet. Bei K3a, nach `k2_sets`, werden für 256 festgelegte Requests (128 S3, 64 A-det, 64 NE) die Zustände am letzten Prompttoken aus allen 28 Blöcken als Float32 gespeichert (etwa 56 MiB). Die Vorbereitung schneidet keine Modellzustände mit. `HIDDEN_STATES_MITSCHNEIDEN = False` schaltet den zusätzlichen Mitschnitt aus. Bereits erledigte Schritte werden dafür nicht erneut gerechnet. Vor der tatsächlichen K3a-Generierung prüft der Lauf Logit-Gleichheit und zusätzlichen Aufwand; die Messung steht unter `observations/hidden_states_v1/benchmark.json`.

Schreibt in Konto B über `dsse_runs_B/variants/htq_v2`; die ursprünglichen B1/B2-Notebooks dürfen nicht parallel laufen.

**Eine GPU:** START → B2_in_A → B1_in_A → A1_DC_zuerst → A2 → ANA. L-CAL separat auf CPU mit hohem RAM; `l_cal.json` vor ANA übernehmen. Danach die finanzierten Zusatzläufe und den Zusatzbericht abschließen. Höchstens zwei Laufzeiten je Konto gleichzeitig.

**Ausführen:** Laufzeittyp einstellen, dann „Alle ausführen“. Bei Abbruch dasselbe H_tq-Notebook erneut ausführen; erfolgreiche Variantenschritte werden übersprungen. Ergebnisse liegen unter `dsse_runs/variants/htq_v2`.


In [ ]:
T0 = __import__('time').time()
SLOT, KONTO = "B1", "B"
PILOT = False             # Pilot-Slot (vor dem Einfrieren)
BRANCH = "codex/r1b-htq-continuation"
HIDDEN_STATES_MITSCHNEIDEN = True   # Optionaler Mitschnitt bei K3a; 256 Requests, ca. 56 MiB
SELBSTTEST = False           # MOCK-Selbsttest dieses Slots vor dem Hauptlauf (einige Minuten; einmal je Code-Stand)
NUR_SELBSTTEST = False       # True: nur einrichten und testen, kein Hauptlauf
WARTEN_MAX_H = 10            # so lange wartet eine Sitzung höchstens auf ihre Startbedingungen
ABFRAGE_S = 120              # Abstand der Drive-Abfragen in Sekunden
LAUFZEIT_MAX_H = 23          # keine Sitzung startet, deren geplante Dauer über diese Laufzeit hinausreicht
import os, sys, json, shutil, subprocess, hashlib, lzma, time
import torch
assert torch.cuda.is_available(), "Keine GPU: Laufzeit → Laufzeittyp ändern → A100 GPU, dann neu starten."
GPU, CC = torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0)
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {GPU}, sm{CC[0]}{CC[1]}, {GB:.0f} GB, torch {torch.__version__}")
assert "A100" in GPU and CC == (8, 0) and GB >= 39, f"SG2: nur eine A100 mit 40 oder 80 GB ist zulässig (hier {GPU})."


In [ ]:
# Getrennte Variantenordner; originale Ergebnisordner werden nur gelesen.
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = "/content/drive/MyDrive"
SOURCE_A = f"{MYDRIVE}/dsse_runs"
SOURCE_B = f"{MYDRIVE}/dsse_runs_B"
assert os.path.isdir(SOURCE_A), "dsse_runs von Konto A fehlt."
assert os.path.isdir(SOURCE_B), "Verknüpfung dsse_runs_B fehlt (Bearbeiterrecht für Konto A nötig)."
SOURCE_RUNS = SOURCE_B
RUNS = os.path.join(SOURCE_RUNS, "variants/htq_v2")
WORK, ROOT, REPO = "/content/work_htq_v2", "/content/root", "/content/DSSE"
os.makedirs(WORK, exist_ok=True); os.makedirs(ROOT, exist_ok=True)
# Nur für reale Unterprozesse; MOCK-Selbsttests erhalten keine Drive-/Variantenvariablen.
_variant_keys = ("R1B_H_POLICY", "R1B_RUNS_B", "R1B_XFER_A", "R1B_SOURCE_RUNS", "R1B_CAPTURE_HIDDEN")
TEST_ENV = {k: v for k, v in os.environ.items() if k not in _variant_keys}
VARIANT_ENV = dict(TEST_ENV, R1B_H_POLICY="htq_v2")
B_RUNS = os.path.join(SOURCE_B, "variants/htq_v2")
LINK = f"{SOURCE_A}/r1b_xfer"
VARIANT_ENV.update(R1B_RUNS_B=B_RUNS, R1B_XFER_A=LINK)
print("H_tq-Ergebnisse:", RUNS)
print("Unveränderte R_S0-Exporte:", LINK)


In [ ]:
# Code: analysis/ju_space (mit round1b/ und round2/), der Plan und die Runde-2-Ergebnisse; ohne --depth, denn der Runner prüft
# den Einfrier-Commit (SG1) und den Code der Runde-1-Teile an ihren Commits (SG3)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "transformers==5.22.2"], check=True)
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--sparse", "-b", BRANCH, "https://github.com/Erikiss/DSSE", REPO], check=True)
subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", "analysis/ju_space", "results/ju_space/plans",
                "results/ju_space/runde2_ergebnisse"], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %cd %s"], capture_output=True, text=True).stdout.strip())
A = f"{REPO}/analysis/ju_space"; R1B = f"{A}/round1b"; D = f"{A}/gpu/data"
FZ_PATH = f"{REPO}/results/ju_space/plans/runde1b/runde1b_freeze.json"
PINS_PATH = f"{REPO}/results/ju_space/plans/runde1b/runde1b_pilot_pins.json"    # vor dem Einfrieren: Pins für die Piloten
FZ = json.load(open(FZ_PATH)) if os.path.exists(FZ_PATH) else {}
if not FZ and PILOT and os.path.exists(PINS_PATH):
    FZ = json.load(open(PINS_PATH))
print("Einfrier-Datensatz:", "vorhanden" if os.path.exists(FZ_PATH) else ("fehlt; Pilot-Pins (Modelle, Datensätze, J-Lenses) aus runde1b_pilot_pins.json"
      if FZ else "fehlt (nur die Piloten r1b_P0 / r1b_P0b laufen vor dem Einfrieren)"))

In [ ]:
# Varianten-Bootstrap / Prüfung vor jedem realen Kopier- oder Hauptlauf.
# Variantenwerkzeug: Ausgabe und Fehler mit Konto sichtbar streamen.
import textwrap
VARIANT_TOOL = f"{R1B}/r1b_htq_variant.py"
def run_variant(action, path, account):
    def show(message):
        for line in str(message).splitlines() or [""]:
            for wrapped in textwrap.wrap(line, width=72, replace_whitespace=False,
                                         drop_whitespace=False, break_on_hyphens=False) or [""]:
                print(wrapped, flush=True)
    show(f"[Konto {account}] {action}: {path}")
    command = [sys.executable, "-u", VARIANT_TOOL, action, path, account]
    try:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=TEST_ENV)
    except OSError as error:
        show(f"[Konto {account}] {action}: Start fehlgeschlagen: {error}")
        raise RuntimeError(f"Konto {account}: {action} konnte nicht starten.") from error
    try:
        for line in process.stdout:
            show(line.rstrip("\r\n"))
        code = process.wait()
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        show(f"[Konto {account}] {action}: unterbrochen; Kindprozess beendet.")
        raise
    finally:
        process.stdout.close()
    show(f"[Konto {account}] {action}: Exit {code}")
    if code:
        raise RuntimeError(f"Konto {account}: {action} fehlgeschlagen (Exit {code}). Details oben.")
    return code

def check_variant(path, account):
    return run_variant("check", path, account)
if not NUR_SELBSTTEST:
    for original, account in ((SOURCE_A, "A"), (SOURCE_B, "B")):
        run_variant("seed", original, account)
        check_variant(os.path.join(original, "variants/htq_v2"), account)
    paired = dict(variant="htq_v2", accounts=["A", "B"], bootstrap_commit=COMMIT)
    for path in (os.path.join(SOURCE_A, "variants/htq_v2"), B_RUNS):
        marker = os.path.join(path, "_htq_pair_ready.json")
        with open(marker + ".tmp", "w") as f:
            json.dump(paired, f, indent=1)
        os.replace(marker + ".tmp", marker)
    print('Beide Variantenordner bereit. Nun wird nur K_S1 neu gerechnet.', flush=True)


In [ ]:
# Eingaben: Modelle zur festen Revision mit Prüfsummen (r1b_core.ensure_model; models/verified.json), für Teil R die Persona-Sätze
# und Runde-1-Daten (wie Runde 2), die J-Lens (Prüfsumme festgelegt); Datensätze lädt jeder Schritt selbst zur festen Revision.
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()

MODELLE = ['qwen3-1.7b']
if MODELLE and not NUR_SELBSTTEST:
    subprocess.run([sys.executable, f"{R1B}/r1b_core.py", "fetch-models", ROOT] + MODELLE, check=True)
if PILOT and not NUR_SELBSTTEST:              # Pilot: PG19-Pilotbücher (GCS, sha256 je Datei) und WikiText-103 (Hugging Face) vorab
    subprocess.run([sys.executable, f"{R1B}/r1b_core.py", "fetch-datasets", ROOT, "pg19", "wikitext103"], check=False)
if SELBSTTEST:                                # der MOCK-Selbsttest braucht nur Konfiguration und Tokenizer von Qwen3-1.7B
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-1.7B", revision="70d244cc86ccca08cf5af4e1e306ecf908b1ad5e", local_dir=f"{ROOT}/models/qwen3-1.7b",
                      allow_patterns=["*.json", "merges.txt", "vocab.json"])
if False:                                # rk_loop-Pfade von Teil R (r2_rk.open_block): Persona-Sätze und Runde-1-Daten
    from huggingface_hub import hf_hub_download
    SUMS = json.load(open(f"{D}/checksums.json")); P = f"{ROOT}/rsi_dl/persona"; os.makedirs(P, exist_ok=True)
    for n, h in SUMS["persona_git_sha1"].items():
        data = open(hf_hub_download(SUMS["persona_repo"], f"persona/{n}.jsonl", repo_type="dataset", revision=SUMS["persona_revision"]), "rb").read()
        assert hashlib.sha1(b"blob %d\0" % len(data) + data).hexdigest() == h, f"Prüfsumme stimmt nicht: {n}"
        open(f"{P}/{n}.jsonl", "wb").write(data)
    for src, dst in SUMS["inputs"].items():
        os.makedirs(os.path.dirname(f"{ROOT}/{dst['path']}"), exist_ok=True)
        raw = lzma.open(f"{D}/{src}").read() if src.endswith(".xz") else open(f"{D}/{src}", "rb").read()
        open(f"{ROOT}/{dst['path']}", "wb").write(raw)
        assert sha256(f"{ROOT}/{dst['path']}") == dst["sha256"], f"Prüfsumme stimmt nicht: {dst['path']}"
if True and not NUR_SELBSTTEST:          # SO-JLENS: aus dsse_runs/r1b_inputs (einmal je Konto hochladen) oder von der URL im Einfrier-Datensatz
    PIN = "6fcc79011bd921ffd87612255e2e99950a124fa519470ee44ebaf161c39be9d6"
    ZIEL = f"{ROOT}/jlens/Qwen3-1.7B_jacobian_lens.pt"; os.makedirs(os.path.dirname(ZIEL), exist_ok=True)
    QUELLE = f"{RUNS}/r1b_inputs/Qwen3-1.7B_jacobian_lens.pt"
    if not os.path.exists(ZIEL) and os.path.exists(QUELLE):
        shutil.copyfile(QUELLE, ZIEL)
    elif not os.path.exists(ZIEL) and (FZ.get("jlens") or {}).get("url"):
        import urllib.request; urllib.request.urlretrieve(FZ["jlens"]["url"], ZIEL)
    if not os.path.exists(ZIEL):                # sonst direkt von Hugging Face (Neuronpedia), dieselbe Datei, Prüfsumme unten
        from huggingface_hub import hf_hub_download
        shutil.copyfile(hf_hub_download("neuronpedia/jacobian-lens", "qwen3-1.7b/jlens/Salesforce-wikitext/Qwen3-1.7B_jacobian_lens.pt"), ZIEL)
    if os.path.exists(ZIEL):
        assert sha256(ZIEL) == PIN, "J-Lens: Prüfsumme stimmt nicht (SO-JLENS)"
        print("J-Lens geprüft.")
    else:
        print("J-Lens FEHLT: dsse_runs/r1b_inputs/Qwen3-1.7B_jacobian_lens.pt hochladen; Schritte, die sie brauchen, scheitern sonst.")
if False and not NUR_SELBSTTEST:   # Teil K (K5a, RK-2): J-Lens von Qwen3.5-2B (Name, sha256: Einfrier-Datensatz jlens.qwen3.5-2b)
    J2 = (FZ.get("jlens") or {}).get("qwen3.5-2b") or {}
    if J2.get("name"):
        ZIEL2, QUELLE2 = f"{ROOT}/jlens/{J2['name']}", f"{RUNS}/r1b_inputs/{J2['name']}"
        os.makedirs(os.path.dirname(ZIEL2), exist_ok=True)
        if not os.path.exists(ZIEL2) and os.path.exists(QUELLE2):
            shutil.copyfile(QUELLE2, ZIEL2)
        elif not os.path.exists(ZIEL2) and J2.get("url"):      # sonst von der URL im Einfrier-Datensatz (Hugging Face, Neuronpedia)
            import urllib.request; urllib.request.urlretrieve(J2["url"], ZIEL2 + ".part"); os.replace(ZIEL2 + ".part", ZIEL2)
        if not os.path.exists(ZIEL2):
            from huggingface_hub import hf_hub_download
            shutil.copyfile(hf_hub_download("neuronpedia/jacobian-lens", "qwen3.5-2b/jlens/Salesforce-wikitext/" + J2["name"]), ZIEL2)
        OK2 = os.path.exists(ZIEL2) and sha256(ZIEL2) == J2.get("sha256")
        if os.path.exists(ZIEL2) and not OK2:
            os.remove(ZIEL2)                            # falsche Prüfsumme: Datei verwerfen
        print("J-Lens Qwen3.5-2B:", "geprüft" if OK2 else "FEHLT oder Prüfsumme falsch (K5a / RK-2 lesen dann 'not run')")
print("Eingaben bereit:", ", ".join(MODELLE) or "keine Modelle nötig")

In [ ]:
# HTQ_START rechnet nur K_S1; kein vorgezogener Kopierlauf des gesamten B1-Slots.


In [ ]:
# Selbsttest (nur Code): alle Sitzungen dieses Slots und ihre Vorgänger im MOCK-Modus auf einem Schein-Drive (zwei Konten, Verknüpfungen
# als symbolische Links), über denselben Runner wie der Hauptlauf; nichts geht ins echte Drive. Marke je Code-Stand in dsse_runs.
MARKE = f"{RUNS}/r1b_selftest_{SLOT}_{COMMIT[:12]}.json"
if not SELBSTTEST:
    print("Selbsttest ausgelassen (SELBSTTEST = False).")
elif os.path.exists(MARKE):
    print("Selbsttest für diesen Code-Stand schon bestanden:", json.load(open(MARKE)))
else:
    t = time.time(); ST = "/content/r1b_selftest"; shutil.rmtree(ST, ignore_errors=True)
    r = subprocess.run([sys.executable, "-u", f"{R1B}/r1b_run_slot.py", SLOT, ROOT, ST, "--selftest"],
                       env=dict(TEST_ENV, RUN_DEVICE="cuda", NT="4"))
    if r.returncode:
        raise SystemExit(f"Selbsttest fehlgeschlagen (Protokolle unter {ST}/work/*/*/logs).")
    json.dump(dict(commit=COMMIT, seconds=round(time.time() - t)), open(MARKE, "w"))
    print(f"Selbsttest bestanden ({time.time() - t:.0f}s).")

In [ ]:
# Ein Mitschnitt gilt nur mit passendem K3a-Schritt als bestätigt.
def hidden_capture_status(runs):
    import hashlib
    root = os.path.join(runs, "r1b_K_S1")
    manifest_path = os.path.join(root, "observations", "hidden_states_v1", "manifest.json")
    result = dict(path=manifest_path, complete=False)
    if not os.path.isfile(manifest_path):
        return dict(result, reason="Manifest fehlt; die Aufzeichnung beginnt erst bei K3a.")
    try:
        with open(manifest_path, encoding="utf-8") as f:
            manifest = json.load(f)
        with open(os.path.join(root, "steps", "k3a.json"), encoding="utf-8") as f:
            step = json.load(f)
    except (OSError, ValueError) as error:
        return dict(result, reason="Manifest oder K3a-Abschluss ist nicht lesbar: " + str(error))
    if not isinstance(manifest, dict) or not isinstance(step, dict):
        return dict(result, reason="Manifest oder K3a-Abschluss hat ein unbekanntes Format.")
    if manifest.get("schema") != "prefill_v1" or manifest.get("status") != "complete" or manifest.get("request_count") != 256:
        return dict(result, reason="Schema, Abschlussstatus oder Requestzahl ist nicht bestätigt.")
    if manifest.get("layer_ids") != list(range(28)):
        return dict(result, reason="Die 28 erwarteten Modellblöcke sind nicht bestätigt.")
    provenance = manifest.get("provenance") or {}
    if not isinstance(provenance, dict) or step.get("status") != "ok" or step.get("name") != "k3a":
        return dict(result, reason="Der zugehörige erfolgreiche K3a-Schritt ist nicht bestätigt.")
    if provenance.get("session") != "r1b_K_S1" or provenance.get("step") != "k3a":
        return dict(result, reason="Der Mitschnitt ist keiner passenden K3a-Sitzung zugeordnet.")
    if any(not provenance.get(key) or provenance.get(key) != step.get(key) for key in ("attempt", "code_commit")):
        return dict(result, reason="Mitschnitt und K3a-Schritt gehören nicht nachweislich zum selben Versuch und Code.")
    families, outputs = manifest.get("families"), step.get("outputs")
    expected_counts = {"s3": 128, "adet": 64, "ne": 64}
    if not isinstance(families, dict) or set(families) != set(expected_counts) or not isinstance(outputs, dict):
        return dict(result, reason="Die drei erwarteten Artefakte oder ihre Schrittprüfsummen fehlen.")
    def digest(path):
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for block in iter(lambda: f.read(8 << 20), b""):
                h.update(block)
        return h.hexdigest()
    directory = "observations/hidden_states_v1"
    try:
        for family, count in expected_counts.items():
            rel = f"{directory}/{family}.pt"
            saved = families[family]
            record = outputs.get(rel)
            if not isinstance(saved, dict) or saved.get("file") != rel or saved.get("count") != count or saved.get("shape") != [count, 28, 2048]:
                return dict(result, reason=f"Unerwarteter Pfad, Umfang oder Tensorform für {family}.")
            expected = saved.get("sha256")
            if not isinstance(record, dict) or not expected or record.get("sha256") != expected:
                return dict(result, reason=f"Manifest und K3a-Prüfsumme stimmen für {family} nicht überein.")
            path = os.path.join(root, rel)
            if not os.path.isfile(path) or digest(path) != expected:
                return dict(result, reason=f"Artefakt {family} fehlt in Drive oder seine Prüfsumme stimmt nicht.")
        benchmark_rel = f"{directory}/benchmark.json"
        if manifest.get("benchmark") != benchmark_rel:
            return dict(result, reason="Der erwartete GPU-Prüfbericht ist nicht zugeordnet.")
        benchmark_path = os.path.join(root, benchmark_rel)
        with open(benchmark_path, encoding="utf-8") as f:
            benchmark = json.load(f)
        if not isinstance(benchmark, dict) or benchmark.get("schema") != "prefill_v1" or benchmark.get("status") != "passed" or benchmark.get("logits_exact") is not True:
            return dict(result, reason="Die GPU-Prüfung bestätigt keine exakte Logit-Gleichheit.")
        benchmark_provenance = benchmark.get("provenance") or {}
        if not isinstance(benchmark_provenance, dict) or any(benchmark_provenance.get(key) != provenance.get(key) for key in ("session", "step", "attempt", "code_commit")):
            return dict(result, reason="GPU-Prüfbericht und Mitschnitt gehören nicht zum selben K3a-Versuch.")
        benchmark_record = outputs.get(benchmark_rel)
        if not isinstance(benchmark_record, dict) or digest(benchmark_path) != benchmark_record.get("sha256"):
            return dict(result, reason="Die Schrittprüfsumme des GPU-Prüfberichts stimmt nicht.")
    except (OSError, ValueError) as error:
        return dict(result, reason="Artefaktprüfung in Drive fehlgeschlagen: " + str(error))
    return dict(result, complete=True, reason="256 Requests, 28 Blöcke, passende K3a-/GPU-Prüfung und drei geprüfte Drive-Artefakte.")
# Nur das neue K_S1; anschließend wird die GPU für B2 frei.
MELDUNG = {0: "H_tq-K_S1 fertig. Als Nächstes HTQ_B2_in_A starten.",
           1: "K_S1 fehlgeschlagen; Variantenprotokolle prüfen und dieses Notebook erneut ausführen.",
           2: "Eine Freigabe ist nicht erfüllt; manifest.json prüfen.",
           3: "Laufzeitgrenze erreicht; HTQ_START in frischer Laufzeit erneut ausführen.",
           4: "Startbedingungen fehlen.", 5: "K_S1 wurde nicht gerechnet; Protokolle prüfen."}
RC = None
if NUR_SELBSTTEST:
    print("NUR_SELBSTTEST = True: kein Hauptlauf.")
else:
    check_variant(RUNS, "B")
    env = dict(VARIANT_ENV, R1B_ACCOUNT="B", R1B_SLOT="B1", R1B_ROOT=ROOT,
               RUN_DEVICE="cuda", NT="8", R1B_RUNTIME_T0=repr(T0),
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    env.pop("R1B_CAPTURE_HIDDEN", None)
    if HIDDEN_STATES_MITSCHNEIDEN:
        env["R1B_CAPTURE_HIDDEN"] = "1"
    cmd = [sys.executable, "-u", f"{R1B}/r1b_run_session.py", "r1b_K_S1", ROOT, f"{WORK}/r1b_K_S1", RUNS]
    while True:
        if time.time() - T0 >= LAUFZEIT_MAX_H * 3600:
            RC = 3; break
        RC = subprocess.run(cmd, env=dict(env, R1B_T0=repr(time.time()))).returncode
        if RC == 3:
            print("Sitzungszeitgrenze: K_S1 wird aus den Variantenzwischenständen fortgesetzt.")
            continue
        break
    if RC == 0:
        assert os.path.isfile(f"{RUNS}/r1b_K_S1/session_done.json"), "K_S1 hat keine fertige Sitzungsmarke geschrieben."
        with open(f"{RUNS}/r1b_K_S1/hn/hn_set.json") as f:
            hs = json.load(f)
        assert hs.get("H_op_name") in ("H_tq", "H_ar"), "K_S1 hat unerwartet H+ als operative Menge verwendet."
        print("Neue operative Menge:", hs["H_op_name"], "mit", len(hs.get("H_op") or []), "Neuronen.")
    print(MELDUNG.get(RC, f"Unerwarteter Rückgabewert {RC}"))
    if HIDDEN_STATES_MITSCHNEIDEN:
        observation = hidden_capture_status(RUNS)
        print("Hidden-State-Manifest:", observation["path"])
        print("Mitschnitt bestätigt vollständig." if observation["complete"] else "Mitschnitt noch nicht bestätigt:", observation["reason"])
        if not observation["complete"]:
            print("Fertige Schritte werden für einen fehlenden Mitschnitt nicht erneut gerechnet.")
        print("GPU-Prüfung und Zusatzaufwand: r1b_K_S1/observations/hidden_states_v1/benchmark.json")
    else:
        print("Zusätzlicher Hidden-State-Mitschnitt ist ausgeschaltet.")


In [ ]:
# Übersicht: Zustand jeder Sitzung des Slots in Drive
for s in ['r1b_K_S1']:
    base = f"{RUNS}/{s}"
    for name in ("session_done.json", "session_partial.json", "session_not_run.json", "start_check.json"):
        if os.path.exists(f"{base}/{name}"):
            S = json.load(open(f"{base}/{name}")); print(f"{s}: {name}", S.get("reason") or "")
            for k, v in (S.get("steps") or {}).items():
                sec = f"{v['seconds']:.0f}s" if isinstance(v.get("seconds"), (int, float)) else ""
                print(f"    {k:<28} {v['state']:<9} {sec:>6}  {(v.get('note') or '')[:80]}")
            break
    else:
        print(f"{s}: noch nicht begonnen")

In [ ]:
# Abschluss: Drive schreiben und trennen
drive.flush_and_unmount()
print("Fertig. Ergebnisse in Google Drive unter dsse_runs/variants/htq_v2/<Sitzung>.")
if RC not in (None, 0):
    raise SystemExit(MELDUNG.get(RC, f"Rückgabewert {RC}"))